<a href="https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Asif-Ahmed-Rezvi/flyrank-internship-ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task type: ranking / scoring. The decision from Week 1 is which pages an editor should review first, so the output should be an ordered queue rather than a yes/no label for every page. I would score each page by its estimated risk of a future, observed decline and use the score to prioritize a limited review capacity.

This is a ranking problem even if a classifier is used underneath: the model is useful only if it produces a better ordering of pages for the editor. The first comparison should remain a transparent fixed-rule baseline.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
from pathlib import Path
import pandas as pd

REPO_DIR = Path("/content/flyrank-internship-ml")
REPO_URL = "https://github.com/Asif-Ahmed-Rezvi/flyrank-internship-ml"

# Clone the repository in Colab if it is not already present
if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}

# Move into the repository
os.chdir(REPO_DIR)

print("Repository ready.")
print("Working directory:", Path.cwd())

# Load the dataset once for the whole notebook
DATA_PATH = REPO_DIR / "data" / "raw" / "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

print(f"Dataset loaded: {len(df):,} rows × {len(df.columns):,} columns")

Repository ready.
Working directory: /content/flyrank-internship-ml
Dataset loaded: 30,000 rows × 44 columns


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Final target: an observed future-decline outcome measured after the decision point. Using the warehouse, I would define a page as positive when its next 30 days of impressions are more than 20% below the immediately preceding 30 days, with features calculated only from information available before that future window. This makes the target an observed outcome rather than a product rule copied from the current snapshot.

Starter-data proxy: the current trend_direction == "down" field can be used only as a sanity-check proxy because it is itself defined from the current two-window comparison. It is not a clean future target, so I will not present it as evidence that the model can predict future decline. trend_direction and trend_pct must also stay out of the feature set.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).

current_decline_proxy = df["trend_direction"].eq("down")

print(
    f"Current decline proxy: "
    f"{current_decline_proxy.sum():,} / {len(df):,} pages "
    f"({current_decline_proxy.mean():.1%})"
)

print("Future 30-day target available in starter snapshot: no")
print(
    "Leakage check: trend_direction and trend_pct "
    "will be excluded from model features."
)


Current decline proxy: 16,262 / 30,000 pages (54.2%)
Future 30-day target available in starter snapshot: no
Leakage check: trend_direction and trend_pct will be excluded from model features.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary metric: Precision@20. The intended user reviews a short queue, so the most important question is: of the 20 pages ranked highest, how many are actually observed to decline in the future target window?

A result is good when the learned ranking has higher Precision@20 than the transparent baseline on an honest holdout period using the same target and the same top-20 review capacity. I would not choose a numeric threshold before seeing the baseline because the baseline defines the practical starting point.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_capacity = 20
print(f"Primary evaluation metric: Precision@{review_capacity}")
print("Success criterion: beat the fixed-rule baseline on the same held-out future period.")


Primary evaluation metric: Precision@20
Success criterion: beat the fixed-rule baseline on the same held-out future period.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: one pseudonymized content item/page. The starter dataset has one row per content item. For this framing exercise I will keep the full page-level table and inspect only the observable fields that could be available before the future target window.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

lane_columns = [
    "content_type",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "engagement_rate",
]

lane_df = df[lane_columns].copy()

print("Shape:", lane_df.shape)
print("One row = one content item/page")
print("Unique page IDs in source:", df["content_id"].nunique())
print("Average position coded as 0 (no position data):", (df["avg_position"] == 0).sum())
print("\nPreview of the lane slice:")
print(lane_df.head(5).to_string(index=False))


Shape: (30000, 9)
One row = one content item/page
Unique page IDs in source: 30000
Average position coded as 0 (no position data): 1205

Preview of the lane slice:
   content_type  impressions_90d  clicks_90d  sessions_90d  content_age_days  days_since_last_update  ctr  avg_position  engagement_rate
keyword article             3803          29            17               187                      20 0.76          10.6             5.88
keyword article            15320           7             9               445                      25 0.05          20.3             0.00
keyword article            12581          11            11               141                      20 0.09          36.5             0.00
keyword article            11751          58            78               463                      22 0.49           6.2             1.28
keyword article            19140          24           145               263                      14 0.13          44.0             0.00


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can capture obvious cases such as a visible page with a large decline, but the review decision can depend on several signals at once: search demand, impressions, position, CTR, content age, freshness, content type, and engagement. The useful pattern may be in combinations or non-linear interactions rather than one universal cutoff.

That is a hypothesis to test, not a result I can claim yet. The fixed rule remains the baseline, and ML earns its place only if it improves Precision@20 on an honest holdout without using future-window fields or leakage.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

signal_summary = pd.DataFrame({
    "signal": [
        "content_type", "impressions_90d", "avg_position",
        "ctr", "content_age_days", "days_since_last_update",
        "engagement_rate"
    ],
    "distinct_or_missing": [
        df["content_type"].nunique(),
        df["impressions_90d"].nunique(),
        df["avg_position"].nunique(),
        df["ctr"].nunique(),
        df["content_age_days"].nunique(),
        df["days_since_last_update"].nunique(),
        df["engagement_rate"].nunique(),
    ]
})

print("Observable signals have multiple values and can contribute different pieces of evidence:")
print(signal_summary.to_string(index=False))
print("\nThis supports testing a multi-signal model, not claiming in advance that ML will win.")


Observable signals have multiple values and can contribute different pieces of evidence:
                signal  distinct_or_missing
          content_type                    3
       impressions_90d                 9438
          avg_position                  869
                   ctr                  401
      content_age_days                  225
days_since_last_update                   57
       engagement_rate                  915

This supports testing a multi-signal model, not claiming in advance that ML will win.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.